<a href="https://colab.research.google.com/github/WardenOfTheCove/VanSolar/blob/main/solar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solar Radiation Prediction Model

This is a Linear Regression Model that takes free datasets provided by NASA, and uses them to predict the day after the last day in the dataset's solar radiation.

In [1]:
#Import necessary libraries and tools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from datetime import date

#Declare variable with path to dataset
file_path = "/content/POWER_Point_Daily_19900101_20100101_049d25N_0123d14W_LST.csv"

def preprocess_data(file_path):
    #Declare variable with dataset's information
    df = pd.read_csv(file_path, skiprows=19)

    #Shifts data up/left and creates Target Column
    df["target"] = df["ALLSKY_SFC_SW_DWN"].shift(-1)


    #Prints the last 5 rows of the dataset
    print("Data Preview with Target Column:")
    print(df[["ALLSKY_SFC_SW_DWN", "target"]].tail)

    df = df.replace(-999.0, np.nan)

    #Clean up the dataset
    df_clean = df.dropna(subset = ["ALLSKY_SFC_SW_DWN", "target"])

    y = df_clean["target"].values
    X = df_clean[["ALLSKY_SFC_SW_DWN"]].values

    return df_clean


def predictionSet():
    df_clean = preprocess_data(file_path)

    latest_row = df_clean.iloc[-1]

    year = int(latest_row['YEAR'])
    month = int(latest_row['MO'])
    day = int(latest_row['DY'])
    latest_date = f"{year:04d}-{month:02d}-{day:02d}"
    tomorrow = date.today() + pd.Timedelta(days=1)
    latest_solar_val = latest_row['ALLSKY_SFC_SW_DWN']

    return latest_date, latest_solar_val, tomorrow

def predictionSet_updated(df_clean):
    # Ensure we have a proper datetime column
    df_clean = df_clean.copy()
    df_clean['DATE'] = pd.to_datetime(df_clean[['YEAR', 'MO', 'DY']].rename(columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day'}))

    # Get real-time calendar date (today)
    today_date = date.today()
    today_dt = pd.to_datetime(today_date)

    # Filter dataset for entries corresponding to the dynamic current date
    matched_row = df_clean[df_clean['DATE'] == today_dt]

    if matched_row.empty:
        # Fallback to the latest available date in dataset if today is not present
        matched_row = df_clean.iloc[[-1]]
        print(f"Warning: Current date {today_dt.strftime('%Y-%m-%d')} not found in dataset. Falling back to latest available record.")

    matched_date_str = matched_row['DATE'].dt.strftime('%Y-%m-%d').values[0]
    latest_solar_val = matched_row['ALLSKY_SFC_SW_DWN'].values[0]
    tomorrow = pd.to_datetime(matched_date_str) + pd.Timedelta(days=1)

    return matched_date_str, latest_solar_val, tomorrow.date()

preprocess_data(file_path)

Data Preview with Target Column:
<bound method NDFrame.tail of       ALLSKY_SFC_SW_DWN    target
0                0.4985    0.4195
1                0.4195    0.3389
2                0.3389    0.3761
3                0.3761 -999.0000
4             -999.0000    0.4577
...                 ...       ...
7301             0.7536    0.7034
7302             0.7034    0.4651
7303             0.4651    0.2582
7304             0.2582    0.3910
7305             0.3910       NaN

[7306 rows x 2 columns]>


,YEAR,MO,DY,ALLSKY_SFC_SW_DWN,WS2M,T2M,TS,QV2M,PW,PS,WS10M_RANGE,WS50M_RANGE,T2MDEW,PRECTOTCORR,target
0,1990,1,1,0.4985,2.70,4.54,5.36,4.76,NaN,100.17,1.94,2.58,3.05,2.02,0.4195
1,1990,1,2,0.4195,2.04,3.30,4.63,4.10,NaN,101.13,4.45,6.17,1.11,0.73,0.3389
2,1990,1,3,0.3389,2.53,5.39,5.94,5.37,NaN,101.14,1.97,1.77,4.92,12.21,0.3761
5,1990,1,6,0.4577,3.17,7.40,7.11,5.96,NaN,99.94,3.27,5.53,6.29,17.57,0.2866
6,1990,1,7,0.2866,2.70,8.61,7.71,6.42,NaN,99.09,6.95,10.92,7.26,8.23,0.5746
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7300,2009,12,27,1.3010,2.50,1.23,3.32,3.64,0.56,101.42,2.07,3.03,-0.49,0.04,0.7536
7301,2009,12,28,0.7536,2.56,1.81,3.43,3.62,0.89,101.57,1.49,2.53,-0.56,0.13,0.7034
7302,2009,12,29,0.7034,2.19,3.51,3.72,4.05,0.85,100.58,1.08,1.67,0.89,0.11,0.4651
7303,2009,12,30,0.4651,2.31,3.11,4.17,4.40,0.87,100.75,2.00,3.10,2.06,3.37,0.2582


In [2]:
import joblib
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

file_path=input("File path to NASA dataset: ")

preprocess_data(file_path)

df_clean = preprocess_data(file_path)

X = df_clean[["ALLSKY_SFC_SW_DWN"]].values
y = df_clean["target"].values

model = LinearRegression()

model.fit(X,y)

y_pred = model.predict(X)

joblib.dump(model, "solar_radiation_model.joblib")

print("Model saved as solar_radiation_model.joblib")

File path to NASA dataset: /content/POWER_Point_Daily_19900101_20100101_049d25N_0123d14W_LST.csv
Data Preview with Target Column:
<bound method NDFrame.tail of       ALLSKY_SFC_SW_DWN    target
0                0.4985    0.4195
1                0.4195    0.3389
2                0.3389    0.3761
3                0.3761 -999.0000
4             -999.0000    0.4577
...                 ...       ...
7301             0.7536    0.7034
7302             0.7034    0.4651
7303             0.4651    0.2582
7304             0.2582    0.3910
7305             0.3910       NaN

[7306 rows x 2 columns]>
Data Preview with Target Column:
<bound method NDFrame.tail of       ALLSKY_SFC_SW_DWN    target
0                0.4985    0.4195
1                0.4195    0.3389
2                0.3389    0.3761
3                0.3761 -999.0000
4             -999.0000    0.4577
...                 ...       ...
7301             0.7536    0.7034
7302             0.7034    0.4651
7303             0.4651    0.2582
730

In [3]:
import joblib
import matplotlib.pyplot as plt

file_path="/content/POWER_Point_Daily_19900101_20100101_049d25N_0123d14W_LST.csv"

model = joblib.load("solar_radiation_model.joblib")

preprocess_data(file_path)

df_clean = preprocess_data(file_path)
matched_date, solar_val, tomorrow_val = predictionSet_updated(df_clean)

matched_date, current_solar_val, tomorrow_date = predictionSet_updated(df_clean)

input_data = np.array([[current_solar_val]])

predicted_solar_val = model.predict(input_data)[0]

print(f"\n--- Prediction Results ---")
print(f"Matched Date in Dataset: {matched_date}")
print(f"Solar Radiation Metric (ALLSKY_SFC_SW_DWN): {current_solar_val:.4f} kW-hr/m^2/day")
print(f"Tomorrow's Date: {tomorrow_date.strftime('%Y-%m-%d')}")
print(f"Predicted Solar Radiation for Tomorrow: {predicted_solar_val:.4f} kW-hr/m^2/day")

Data Preview with Target Column:
<bound method NDFrame.tail of       ALLSKY_SFC_SW_DWN    target
0                0.4985    0.4195
1                0.4195    0.3389
2                0.3389    0.3761
3                0.3761 -999.0000
4             -999.0000    0.4577
...                 ...       ...
7301             0.7536    0.7034
7302             0.7034    0.4651
7303             0.4651    0.2582
7304             0.2582    0.3910
7305             0.3910       NaN

[7306 rows x 2 columns]>
Data Preview with Target Column:
<bound method NDFrame.tail of       ALLSKY_SFC_SW_DWN    target
0                0.4985    0.4195
1                0.4195    0.3389
2                0.3389    0.3761
3                0.3761 -999.0000
4             -999.0000    0.4577
...                 ...       ...
7301             0.7536    0.7034
7302             0.7034    0.4651
7303             0.4651    0.2582
7304             0.2582    0.3910
7305             0.3910       NaN

[7306 rows x 2 columns]>

--- P